# Previsão de Direção de Ações com LSTM

Neste trabalho, o objetivo é utilizar uma rede neural do tipo LSTM para prever a direção do preço de uma ação (se vai subir ou cair no próximo dia).

A escolha da LSTM se deve à sua capacidade de trabalhar com dados sequenciais, sendo bastante utilizada em séries temporais como dados financeiros.

Diferente de prever o preço exato, aqui o problema foi tratado como classificação binária:
- 1 → o preço sobe
- 0 → o preço cai

Nesta etapa, são importadas as bibliotecas necessárias para manipulação de dados, coleta de dados financeiros e construção do modelo de deep learning.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn

Os dados foram coletados utilizando a biblioteca yfinance.

Foram utilizadas múltiplas variáveis (Open, High, Low, Close e Volume), pois isso pode ajudar o modelo a capturar melhor os padrões do mercado.

In [2]:
df = yf.download('PETR4.SA', start='2018-01-01', end='2024-01-01')
df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
df.dropna(inplace=True)

df.head()

/tmp/ipykernel_35149/1483309637.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download('PETR4.SA', start='2018-01-01', end='2024-01-01')
[*********************100%***********************]  1 of 1 completed


Price,Open,High,Low,Close,Volume
Ticker,PETR4.SA,PETR4.SA,PETR4.SA,PETR4.SA,PETR4.SA
Date,,,,,
2018-01-02,4.313444,4.409358,4.313444,4.409358,33461800
2018-01-03,4.393372,4.454650,4.361402,4.449322,55940900
2018-01-04,4.470637,4.518593,4.428009,4.457315,37064900
2018-01-05,4.449322,4.491950,4.414686,4.483957,26958200
2018-01-08,4.459979,4.537243,4.451986,4.537243,28400000


O problema foi transformado em classificação.

Se o preço do próximo dia for maior que o atual:
- recebe 1 (subida)

Caso contrário:
- recebe 0 (queda)

In [3]:
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
df.dropna(inplace=True)

A normalização foi aplicada para colocar os dados na mesma escala.

Isso é importante porque redes neurais funcionam melhor com dados normalizados.

In [4]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[['Open', 'High', 'Low', 'Close', 'Volume']])

Como a LSTM trabalha com dados sequenciais, os dados foram transformados em janelas de tempo.

Cada entrada do modelo contém 20 dias anteriores para prever o próximo dia.

In [5]:
def create_sequences(data, target, seq_length):
    X, y = [], []

    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(target[i+seq_length])

    return np.array(X), np.array(y)

seq_length = 20

X, y = create_sequences(scaled_data, df['Target'].values, seq_length)

Os dados foram divididos em treino e teste.

Isso permite avaliar o desempenho do modelo em dados que ele não viu durante o treinamento.

In [6]:
split = int(0.8 * len(X))

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

O PyTorch utiliza tensores, então foi necessário converter os dados para esse formato.

In [7]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # pega o último passo da sequência
        out = self.fc(out)
        out = self.sigmoid(out)
        return out

A rede LSTM foi utilizada para capturar padrões temporais.

A saída passa por uma função sigmoid, pois o problema é de classificação binária.

In [8]:
input_size = X.shape[2]
hidden_size = 50
num_layers = 2

model = LSTMModel(input_size, hidden_size, num_layers)

Foi utilizada a função de perda BCELoss, adequada para classificação binária.

O otimizador Adam foi escolhido por ser eficiente e amplamente utilizado.

In [9]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Durante o treinamento, o modelo ajusta seus pesos com base no erro utilizando backpropagation.

In [10]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [11]:
epochs = 10

for epoch in range(epochs):
    model.train()

    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f'Época {epoch+1}, Loss: {loss.item():.4f}')

Época 1, Loss: 0.6937
Época 2, Loss: 0.6933
Época 3, Loss: 0.6930
Época 4, Loss: 0.6927
Época 5, Loss: 0.6925
Época 6, Loss: 0.6925
Época 7, Loss: 0.6925
Época 8, Loss: 0.6926
Época 9, Loss: 0.6927
Época 10, Loss: 0.6927


A avaliação foi feita utilizando a acurácia, que mede a proporção de acertos do modelo.

In [12]:
model.eval()

with torch.no_grad():
    predictions = model(X_test_t)
    predictions = (predictions > 0.5).float()

    accuracy = (predictions == y_test_t).float().mean()

print(f'Acurácia: {accuracy.item():.4f}')

Acurácia: 0.5612
